In [1]:
%%writefile /home/jovyan/work/README.md
# Olist E-Commerce Data Pipeline

A medallion pipeline (Bronze -> Silver -> Gold) built in Jupyter / PySpark on top of the Blue Owls mock data API. It produces a star schema (1 fact + 3 dimensions) and two Spark SQL analyses.

## How to run

1. `docker-compose up` (starts the mock API and Jupyter Lab).
2. Open http://localhost:8888 and run the notebooks in `work/` in this order:

| Notebook | Purpose |
|---|---|
| `01_explore` | Look at API schemas and one sample page (exploration only) |
| `02_ingestion` | API -> Bronze CSVs (auth, retries, validation, manifest) |
| `03_silver` | Bronze -> Silver (typing, dedupe/upsert, validity flags) |
| `04_gold` | Silver -> Gold star schema + validation checks |
| `05_sql_analysis` | Query 1 (revenue trend), saved to `sql/query_1.sql` |
| `06_sql_query2` | Query 2 (seller scorecard), saved to `sql/query_2.sql` |

Outputs are in `submission/output/{bronze,silver,gold}`, quarantined records in `output/quarantine`, logs in `output/logs`. Bronze is append-only, so the committed `output/manifest.json` must stay together with the committed Bronze files. If you want a completely fresh run, delete the whole `output/` folder first.

`mock-api/` (including its `data/` folder) comes with the assessment template and is what serves the API. It is not the pipeline's own data: the pipeline only reads from the API.

## Part 1: Resilience

The API injects 500s, 429s and token expiry, so all HTTP calls go through one `ApiClient`:

- **401**: log in again and retry the same request (tokens last 15 minutes).
- **429**: wait for the `Retry-After` header plus a random jitter.
- **500 / network error / timeout / invalid JSON**: exponential backoff (1s, 2s, 4s ... capped at 30s) plus jitter, up to 8 attempts.
- **Other 4xx**: fail immediately. These are bugs in our request, so retrying would only waste time.
- If a page still fails after all retries, it is logged and skipped. It is not written to the manifest, so the next run retries it. After 3 failed pages in a row the endpoint stops, to avoid hammering an API that is probably down. One endpoint failing never stops the others.
- **Validation**: every record is checked (is an object, expected columns present, key fields not empty, no nested values). Bad records go to `quarantine/<endpoint>_quarantine.jsonl` with the reason and page number. Nothing is dropped silently. In my run 0 records were quarantined, so that path was written but not exercised by real bad data.

**Decision I thought about: how to make Bronze idempotent.** Options were (a) check existing Bronze rows before writing, or (b) a manifest of finished pages. I chose the manifest: the key is `endpoint | date_from | page_size | page`, and it is written only after the page is safely appended to the CSV (atomic write via temp file + rename). A re-run skips finished pages without calling the API at all. Option (a) would re-download everything and re-read the growing CSV each time. The trade-off: if the notebook crashes between the CSV write and the manifest write, that page is fetched again and duplicated in Bronze. That gives at-least-once delivery, never data loss, and Silver removes such duplicates (see below). I tested it: the first run fetched 3 / 4 / 20 / 7 / 1 / 21 pages (orders, order_items, customers, products, sellers, payments); a second run fetched 0 pages, skipped all of them and wrote 0 rows.

## Part 3: Silver

- Everything is read as text first (no schema guessing, so zip codes keep leading zeros), trimmed, and empty strings become NULL.
- **Upsert**: exact duplicate rows are removed, then only the latest `_ingested_at` per natural key is kept (`order_id`, `order_id + order_item_id`, `product_id`, ...). Silver is rebuilt from Bronze on every run with overwrite, so the same Bronze always gives the same Silver.
- Types: timestamps, `decimal(12,2)` for money, integers. A value that cannot be converted makes the record invalid with the column named in `_invalid_reason`. The raw value is never silently lost.
- `_is_valid` / `_invalid_reason` rules: missing keys, negative price/freight/payment, delivery before purchase, item with no matching order, payment with no matching order.
- NULL strategy: missing delivery dates stay NULL (order not delivered yet); missing price/payment makes the record invalid (no invented money); missing category becomes `unknown`; missing weight/dimensions stay NULL (0 would distort volume).
- Invalid records are kept and flagged, not deleted. Gold only uses valid rows.

## Part 2/3: Gold

- **Surrogate keys** are `xxhash64(natural key)` masked to 63 bits: deterministic across runs, unlike `row_number()`. They are 64-bit (not 32-bit) integers, and uniqueness is checked in the validation step.
- **Foreign keys** in the fact table are taken by joining to the dimensions, so an orphan key cannot exist. A fact row without a dimension match would go to quarantine and be counted (this run: 0).
- **Payment split**: the order's total payment is distributed over its items in proportion to `price`. Rounding to cents can leave the split a few cents off, so the difference is added to the most expensive item. The validation step confirms that every order's split adds up exactly to its payment total.
- `payment_type` = type of the highest payment row (ties: first alphabetically); `payment_installments` = max over payment rows.
- `dim_customers`: one row per `customer_unique_id`; city/state/zip from the most recent order (order_id as deterministic tie-breaker).

## Assumptions and trade-offs

- **Date scope**: the API can date-filter only `orders` and `order_items`. Customers, products, sellers and payments are ingested in full and the cutoff (2018-07-01) is applied by joining to in-scope orders: 90,661 of 103,886 payments and 1,249 of 15,590 order items have no in-scope order and are flagged invalid in Silver. The likely cause for the items is that the API filters them on a different date field than the order purchase date (not verified).
- Because of this scope, `total_orders`, `total_spend` and `first_order_date` in `dim_customers` only include orders from 2018-07-01 onward, not the customer's full history.
- `days_to_deliver` and `days_delivery_vs_estimate` are calendar-day differences (`datediff`). All timestamps are treated as UTC with no time-zone conversion.
- Cancelled and unavailable orders are included in revenue, because the assignment does not say to exclude them. An order without payments keeps its items with NULL payment columns.

## Part 4: SQL

- **Query 1**: ranking is computed over all categories in each month, before narrowing to the top 5 categories. Month-over-month growth is NULL when the previous calendar month is missing (no misleading comparison across a gap). **Limitation in the result:** in the Gold fact table there are 7,092 items in July 2018, 7,248 in August, 1 in September and none in October. With the 2018-07-01 cutoff and the required "at least 10 transactions per category-month" filter, only July and August 2018 qualify. So each of the top 5 categories has at most 2 rows, MoM growth exists only for August, and the 3-month rolling average covers at most 2 months. This follows from the data scope, not from the query logic.
- **Query 2**: the item-level fact table is first collapsed to one row per (seller, order), so multi-item orders are counted once. Late rate = late orders / delivered orders (an undelivered order cannot be late). Percentiles use `PERCENT_RANK()` with lower-is-better metrics ranked descending. Ties share the lowest rank, so sellers with a 0% late rate get 74.29 on the on-time percentile instead of 100. `CUME_DIST()` would be the alternative if ties should get the upper rank. 141 sellers have 20 or more orders.
- Both queries were cross-checked against an independent DataFrame / pandas calculation (totals match for Query 1; maximum composite score difference 0.0043 for Query 2, which is only rounding).

## Validation results (this run)

Bronze rows: orders 12,824 / order_items 15,590 / customers 99,441 / products 32,951 / sellers 3,095 / payments 103,886 (0 duplicate keys). Gold rows: dim_customers 12,647 / dim_products 32,951 / dim_sellers 3,095 / fact_order_items 14,341. All checks pass: no orphan keys, unique surrogate keys, unique fact grain, payment split reconciles, CSV row counts match.

## What I would change for production on Azure / Microsoft Fabric

- **Storage and format**: write Delta tables in ADLS Gen2 / OneLake instead of CSV. The Silver upsert becomes a real Delta `MERGE` on the natural key, and Bronze keeps its append-only history.
- **Scheduling**: Fabric Data Factory (or Azure Data Factory) pipeline with a daily trigger, running the notebooks in order with parameters. Incremental loads use a watermark (last ingested date) instead of the whole range.
- **Secrets and security**: the API credentials move into Azure Key Vault, accessed through a managed identity. Nothing is stored in code or environment files. Storage is private (private endpoints, RBAC).
- **Monitoring**: structured logs go to Log Analytics. Alerts fire on failed pages, a rising quarantine count, or row counts dropping versus the previous run. The validation cell becomes automated data-quality checks that fail the pipeline.
- **CI/CD**: GitHub Actions runs unit tests for the retry and validation logic against the mock API, then deploys notebooks to dev, then prod.
- **Cost**: Spark pools auto-pause, with a small node size for this data volume. Only new partitions are processed, and Gold is stored partitioned by `order_date`.

Writing /home/jovyan/work/README.md


In [2]:
%%writefile /home/jovyan/work/requirements.txt
# Packages used by the notebooks. PySpark, Jupyter and the base Python stack are
# already provided by the jupyter/pyspark-notebook Docker image; these are listed
# explicitly for anyone running the code outside the image.
requests
pandas

Writing /home/jovyan/work/requirements.txt
